# XCT Segmentation — U-Net vs. Otsu Benchmark (for the upgrade report)

Compares the trained U-Net against a standard multi-Otsu thresholding baseline
(the same approach used for the first-pass masks in `xct_membrane_segmentation.ipynb`),
on whatever hand-corrected slices you have available.

**Produces, per method (U-Net / Otsu):**

- Per-class and mean IoU, Dice/F1, precision, recall, pixel accuracy
- Aggregate confusion matrices
- Paired statistical tests (Wilcoxon signed-rank + paired t-test) and a
  bootstrap confidence interval on the IoU improvement
- Per-slice and per-class inference timing, throughput, and an extrapolation
  to the full ~4000-slice stack
- Model size, parameter count, and peak memory use
- Bar charts, box plots, confusion-matrix heatmaps, and qualitative overlays
- CSV/Markdown/PNG exports under `output_dir/evaluation_report/`, ready to
  drop into the report

**Pipeline position:** run this after `xct_unet_training.ipynb` has produced a
saved model. It reuses the same `output_dir` (`images/`, `masks_corrected/`)
and re-loads the model from `model_dir`.


## A note on fairness before you run this

The U-Net was trained on some subset of your hand-corrected slices. If you
evaluate it on **all** corrected slices (including ones it trained on), its
scores here will be optimistic compared to how it'll perform on genuinely new
data — Otsu, by contrast, uses no training data at all, so this isn't a fair
fight unless you control for it.

Two ways to handle this:

1. **Best: use a real held-out set.** If you noted down the validation
   `sample_id`s that `xct_unet_training.ipynb` printed when it split the data
   (`"Validation sample_ids: [...]"`), paste them into
   `CONFIG["held_out_sample_ids"]` below. Only those slices will be scored —
   Otsu vs. a U-Net on data it never trained on.
2. **If you didn't save that list:** leave `held_out_sample_ids` empty. This
   notebook will still run, scoring on every corrected slice, but it will
   print a warning and you should flag the caveat in your report — the U-Net
   numbers are a (mild-to-moderate, depending on how much of the ground truth
   set it was trained on) best case, not a generalisation estimate.

Going forward, the cleanest fix is to hold out a handful of corrected slices
from training entirely, specifically to keep as a permanent test set for
future model comparisons like this one.


## 0. Setup

In [ ]:
# pip install numpy pandas matplotlib scipy scikit-image scikit-learn tifffile tqdm psutil
# pip install "tensorflow<2.16" segmentation-models
import os
os.environ["SM_FRAMEWORK"] = "tf.keras"  # must be set BEFORE importing segmentation_models

import json
import time
import platform
import subprocess
from pathlib import Path
from itertools import combinations

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tifffile
import psutil
from tqdm.auto import tqdm

from scipy.ndimage import median_filter
from scipy import stats as sstats
from skimage.filters import threshold_multiotsu
from skimage.morphology import remove_small_objects, remove_small_holes, disk, binary_closing
from matplotlib.colors import to_rgb as mcolor_to_rgb

import tensorflow as tf
import segmentation_models as sm

print("TensorFlow:", tf.__version__)
print("GPUs visible:", tf.config.list_physical_devices("GPU"))


## 1. Config

Mirror the paths/model settings from `xct_unet_training.ipynb` — the normalization and inference code here need to match what the model was trained with.

In [ ]:
CONFIG = {
    # --- same paths as the other notebooks ---
    "output_dir": Path("path/to/output"),      # images/, masks_corrected/, sample_manifest.csv live here
    "model_dir": Path("/path/to/model/output"),
    "model_filename": "best_model.h5",         # or "final_model.h5"

    # --- must match training CONFIG (xct_unet_training.ipynb section 1) ---
    "phase_names": ["voidage", "membrane", "polymer_cartridge"],
    "n_classes": 3,
    "backbone": "resnet34",
    "patch_size": 384,
    "inference_stride_fraction": 0.5,
    "inference_batch_size": 32,

    # --- Otsu baseline params, matching xct_membrane_segmentation.ipynb CONFIG ---
    "denoise_median_size": 3,
    "min_object_size_px": 64,
    "min_hole_size_px": 64,
    "closing_radius": 1,

    # --- evaluation set ---
    # paste the "Validation sample_ids" printed by xct_unet_training.ipynb here for a fair,
    # held-out comparison. Leave empty to evaluate on every corrected slice instead (see the
    # fairness note above -- this notebook will warn you if you do).
    "held_out_sample_ids": [],

    # --- report context ---
    "full_stack_n_slices": 4000,        # for extrapolating per-slice timing to the whole stack
    "manual_training_time_hours": None, # optional: fill in from your training run's wall-clock time

    # --- statistics ---
    "n_bootstrap": 2000,
    "random_seed": 42,

    # --- qualitative overlay colours ---
    # voidage is treated as background and left untinted (plain greyscale) since it's the
    # least interesting phase and was getting visually confused with membrane's colour;
    # membrane/polymer_cartridge get distinct, high-contrast colours instead.
    "overlay_bg_class": 0,               # class value to leave untinted (voidage)
    "overlay_colors": ["red", "cyan"],   # colours for the remaining classes, in class-id order
    "overlay_alpha": 0.45,

    # --- output ---
    "report_dir": None,  # defaults to output_dir / "evaluation_report" if left as None
}

if CONFIG["report_dir"] is None:
    CONFIG["report_dir"] = CONFIG["output_dir"] / "evaluation_report"
CONFIG["report_dir"].mkdir(parents=True, exist_ok=True)

rng = np.random.default_rng(CONFIG["random_seed"])
CONFIG


## 2. Load the manifest and build the evaluation set

In [ ]:
manifest_path = CONFIG["output_dir"] / "sample_manifest.csv"
manifest = pd.read_csv(manifest_path)

def sample_filename(row):
    return f"sample_{row.sample_id:03d}_{Path(row.filename).stem}.tif"

labelled = manifest.loc[manifest.corrected].reset_index(drop=True)
print(f"{len(labelled)} hand-corrected slices available in total")

if CONFIG["held_out_sample_ids"]:
    eval_rows = labelled[labelled.sample_id.isin(CONFIG["held_out_sample_ids"])].reset_index(drop=True)
    missing = set(CONFIG["held_out_sample_ids"]) - set(eval_rows.sample_id)
    if missing:
        print(f"WARNING: sample_id(s) {sorted(missing)} not found among corrected slices -- check the list.")
    print(f"Evaluating on the {len(eval_rows)} specified held-out slice(s) -- fair, held-out comparison.")
else:
    eval_rows = labelled
    print(f"WARNING: no held_out_sample_ids given -- evaluating on all {len(eval_rows)} corrected slice(s), "
          f"which the U-Net may have partly trained on. Treat U-Net numbers below as a best case, not a "
          f"generalisation estimate, and say so in the report.")

assert len(eval_rows) >= 2, "Need at least a couple of corrected slices to evaluate against."
eval_rows[["sample_id", "slice_index", "corrected"]]


## 3. Load images/ground truth, and reconstruct the Otsu baseline

`normalize_to_uint8` needs the same 1st/99th-percentile stretch the U-Net saw during
training. We don't have the exact percentiles saved, so they're recomputed here from
the full corrected set -- this is a stable, data-driven property of the scan and
shouldn't meaningfully change between runs, but note it in the report as an
approximation if you want to be precise.

In [ ]:
def load_image(fname):
    return tifffile.imread(CONFIG["output_dir"] / "images" / fname).astype(np.float32)

def load_gt_mask(fname):
    return tifffile.imread(CONFIG["output_dir"] / "masks_corrected" / fname)

_all_vals = np.concatenate([load_image(sample_filename(row)).ravel() for _, row in labelled.iterrows()])
P_LOW, P_HIGH = np.percentile(_all_vals, [1, 99])
print(f"Normalizing with 1st/99th percentiles: {P_LOW:.1f} / {P_HIGH:.1f}")
del _all_vals

def normalize_to_uint8(img):
    clipped = np.clip(img, P_LOW, P_HIGH)
    scaled = (clipped - P_LOW) / (P_HIGH - P_LOW + 1e-8)
    return (scaled * 255).astype(np.uint8)

def to_rgb(img_uint8):
    return np.stack([img_uint8] * 3, axis=-1)

preprocess_input = sm.get_preprocessing(CONFIG["backbone"])


### Otsu baseline

Same global-threshold approach as `xct_membrane_segmentation.ipynb`: thresholds are
fit once from a pooled histogram (here, over every corrected slice's raw image) and
applied everywhere, followed by the same per-class morphological cleanup.

In [ ]:
def denoise(img):
    size = CONFIG["denoise_median_size"]
    if not size:
        return img
    return median_filter(img, size=size)

_pooled = np.concatenate([denoise(load_image(sample_filename(row))).ravel() for _, row in labelled.iterrows()])
otsu_thresholds = threshold_multiotsu(_pooled, classes=CONFIG["n_classes"])
print("Global Otsu thresholds:", otsu_thresholds)
del _pooled

def cleanup_labels(labels):
    cleaned = np.zeros_like(labels)
    selem = disk(CONFIG["closing_radius"]) if CONFIG["closing_radius"] else None
    for class_id in range(CONFIG["n_classes"]):
        mask = labels == class_id
        mask = remove_small_objects(mask, min_size=CONFIG["min_object_size_px"])
        mask = remove_small_holes(mask, area_threshold=CONFIG["min_hole_size_px"])
        if selem is not None:
            mask = binary_closing(mask, selem)
        cleaned[mask] = class_id
    return cleaned

def segment_otsu(img, thresholds=otsu_thresholds):
    denoised = denoise(img)
    labels = np.digitize(denoised, bins=thresholds).astype(np.uint8)
    return cleanup_labels(labels)

def overlay(img, labels):
    '''Alpha-blend named colours onto foreground classes only; the background
    class (voidage) is left as the literal untouched greyscale pixel -- no
    library defaults or automatic colour choice involved, so there's no risk
    of a stray tint on the class that's supposed to be plain.'''
    img_norm = (img - img.min()) / max(1, (img.max() - img.min()))
    rgb = np.stack([img_norm] * 3, axis=-1).astype(np.float32)

    non_bg_classes = [c for c in range(CONFIG["n_classes"]) if c != CONFIG["overlay_bg_class"]]
    assert len(CONFIG["overlay_colors"]) >= len(non_bg_classes),         "Not enough entries in CONFIG['overlay_colors'] for the non-background classes."

    out = rgb.copy()
    alpha = CONFIG["overlay_alpha"]
    for class_id, color_name in zip(non_bg_classes, CONFIG["overlay_colors"]):
        mask = labels == class_id
        color = np.array(mcolor_to_rgb(color_name), dtype=np.float32)
        out[mask] = (1 - alpha) * rgb[mask] + alpha * color
    return out


## 4. Load the trained U-Net

In [ ]:
model = sm.Unet(
    CONFIG["backbone"],
    classes=CONFIG["n_classes"],
    activation="softmax",
    encoder_weights=None,
    input_shape=(CONFIG["patch_size"], CONFIG["patch_size"], 3),
)
model_path = CONFIG["model_dir"] / CONFIG["model_filename"]
model.load_weights(str(model_path))
print("Loaded weights ->", model_path)

n_params = model.count_params()
model_size_mb = model_path.stat().st_size / (1024 ** 2)
print(f"Parameters: {n_params:,}")
print(f"Model file size: {model_size_mb:.1f} MB")


### Tiled inference (same sliding-window logic as `xct_unet_training.ipynb` section 8)

Kept identical to the training notebook's `predict_full_image` so timing/behaviour here
matches what full-stack segmentation would actually do.

In [ ]:
@tf.function(jit_compile=True)
def _predict_step(x):
    return model(x, training=False)

def _predict_batches(batch, batch_size):
    preds = []
    for i in range(0, len(batch), batch_size):
        chunk = batch[i:i + batch_size]
        pad_n = batch_size - len(chunk)
        if pad_n > 0:
            pad = np.zeros((pad_n, *chunk.shape[1:]), dtype=chunk.dtype)
            chunk = np.concatenate([chunk, pad], axis=0)
        pred = _predict_step(tf.constant(chunk)).numpy()
        if pad_n > 0:
            pred = pred[:-pad_n]
        preds.append(pred)
    return np.concatenate(preds, axis=0)

def segment_unet(img_raw, patch_size=None, stride=None):
    patch_size = patch_size or CONFIG["patch_size"]
    stride = stride or max(1, int(patch_size * CONFIG["inference_stride_fraction"]))

    img_rgb = to_rgb(normalize_to_uint8(img_raw))
    H, W = img_raw.shape

    pad_h = (patch_size - H % stride) % stride
    pad_w = (patch_size - W % stride) % stride
    img_padded = np.pad(img_rgb, ((0, pad_h + patch_size), (0, pad_w + patch_size), (0, 0)), mode="reflect")
    Hp, Wp = img_padded.shape[:2]

    ys = list(range(0, Hp - patch_size + 1, stride))
    xs = list(range(0, Wp - patch_size + 1, stride))

    batch, positions = [], []
    for y in ys:
        for x in xs:
            tile = img_padded[y:y + patch_size, x:x + patch_size]
            batch.append(preprocess_input(tile.astype(np.float32)))
            positions.append((y, x))
    batch = np.stack(batch)

    preds = _predict_batches(batch, CONFIG["inference_batch_size"])

    prob_sum = np.zeros((Hp, Wp, CONFIG["n_classes"]), dtype=np.float32)
    weight = np.zeros((Hp, Wp), dtype=np.float32)
    for (y, x), pred in zip(positions, preds):
        prob_sum[y:y + patch_size, x:x + patch_size] += pred
        weight[y:y + patch_size, x:x + patch_size] += 1

    prob_avg = prob_sum[:H, :W] / np.maximum(weight[:H, :W, None], 1e-8)
    return np.argmax(prob_avg, axis=-1).astype(np.uint8)

# warm up the JIT-compiled graph once (outside the timing loop) so the first real
# slice isn't penalised by XLA's one-off compilation cost
_dummy = np.zeros((CONFIG["patch_size"], CONFIG["patch_size"]), dtype=np.float32)
_ = segment_unet(_dummy)
del _dummy
print("U-Net warmed up.")


## 5. Metrics

Standard per-class overlap metrics computed from a confusion matrix: IoU (Jaccard),
Dice/F1, precision, recall, plus overall pixel accuracy. Aggregate scores are computed
from the **summed** confusion matrix across all evaluated slices (more statistically
stable than averaging per-slice ratios, since it implicitly weights by pixel count),
while per-slice scores are also kept for the paired significance tests below.

In [ ]:
def confusion_matrix(gt, pred, n_classes):
    idx = gt.astype(np.int64) * n_classes + pred.astype(np.int64)
    counts = np.bincount(idx.ravel(), minlength=n_classes * n_classes)
    return counts.reshape(n_classes, n_classes)

def metrics_from_confusion(cm):
    n = cm.shape[0]
    tp = np.diag(cm).astype(np.float64)
    fp = cm.sum(axis=0) - tp
    fn = cm.sum(axis=1) - tp
    iou = tp / np.maximum(tp + fp + fn, 1)
    dice = 2 * tp / np.maximum(2 * tp + fp + fn, 1)
    precision = tp / np.maximum(tp + fp, 1)
    recall = tp / np.maximum(tp + fn, 1)
    pixel_acc = tp.sum() / max(cm.sum(), 1)
    return {
        "iou": iou, "dice": dice, "precision": precision, "recall": recall,
        "mean_iou": iou.mean(), "mean_dice": dice.mean(), "pixel_acc": pixel_acc,
    }


## 6. Timed evaluation loop

Runs both methods on every evaluation slice, timing each call individually (wall-clock,
via `time.perf_counter`) and scoring both against the hand-corrected ground truth.

In [ ]:
per_slice_records = []
cm_otsu_total = np.zeros((CONFIG["n_classes"], CONFIG["n_classes"]), dtype=np.int64)
cm_unet_total = np.zeros((CONFIG["n_classes"], CONFIG["n_classes"]), dtype=np.int64)

qualitative_examples = {}  # sample_id -> dict of arrays, for section 9's overlay panel

for _, row in tqdm(list(eval_rows.iterrows()), desc="Evaluating"):
    fname = sample_filename(row)
    img = load_image(fname)
    gt = load_gt_mask(fname)

    t0 = time.perf_counter()
    pred_otsu = segment_otsu(img)
    t_otsu = time.perf_counter() - t0

    t0 = time.perf_counter()
    pred_unet = segment_unet(img)
    t_unet = time.perf_counter() - t0

    cm_o = confusion_matrix(gt, pred_otsu, CONFIG["n_classes"])
    cm_u = confusion_matrix(gt, pred_unet, CONFIG["n_classes"])
    cm_otsu_total += cm_o
    cm_unet_total += cm_u

    m_o = metrics_from_confusion(cm_o)
    m_u = metrics_from_confusion(cm_u)

    per_slice_records.append({
        "sample_id": row.sample_id, "slice_index": row.slice_index,
        "otsu_mean_iou": m_o["mean_iou"], "unet_mean_iou": m_u["mean_iou"],
        "otsu_pixel_acc": m_o["pixel_acc"], "unet_pixel_acc": m_u["pixel_acc"],
        "otsu_time_s": t_otsu, "unet_time_s": t_unet,
        **{f"otsu_iou_{name}": v for name, v in zip(CONFIG["phase_names"], m_o["iou"])},
        **{f"unet_iou_{name}": v for name, v in zip(CONFIG["phase_names"], m_u["iou"])},
    })

    qualitative_examples[row.sample_id] = dict(img=img, gt=gt, pred_otsu=pred_otsu, pred_unet=pred_unet)

per_slice_df = pd.DataFrame(per_slice_records)
per_slice_df


## 7. Aggregate scores and statistical comparison

In [ ]:
agg_otsu = metrics_from_confusion(cm_otsu_total)
agg_unet = metrics_from_confusion(cm_unet_total)

summary_rows = []
for i, name in enumerate(CONFIG["phase_names"]):
    summary_rows.append({
        "class": name,
        "otsu_iou": agg_otsu["iou"][i], "unet_iou": agg_unet["iou"][i],
        "otsu_dice": agg_otsu["dice"][i], "unet_dice": agg_unet["dice"][i],
        "otsu_precision": agg_otsu["precision"][i], "unet_precision": agg_unet["precision"][i],
        "otsu_recall": agg_otsu["recall"][i], "unet_recall": agg_unet["recall"][i],
    })
summary_rows.append({
    "class": "MEAN / OVERALL",
    "otsu_iou": agg_otsu["mean_iou"], "unet_iou": agg_unet["mean_iou"],
    "otsu_dice": agg_otsu["mean_dice"], "unet_dice": agg_unet["mean_dice"],
    "otsu_precision": np.nan, "unet_precision": np.nan,
    "otsu_recall": np.nan, "unet_recall": np.nan,
})
summary_df = pd.DataFrame(summary_rows)
print(f"Pixel accuracy -- Otsu: {agg_otsu['pixel_acc']:.4f}   U-Net: {agg_unet['pixel_acc']:.4f}")
summary_df.round(4)


### Paired significance tests

Both methods were scored on the *same* slices, so this is a paired comparison:
per-slice mean IoU (U-Net minus Otsu). A Wilcoxon signed-rank test is used as the
primary test (doesn't assume normality, appropriate for small sample counts typical
of a hand-corrected ground-truth set); a paired t-test is reported alongside it as a
supplementary check. A bootstrap CI (resampling slices with replacement) gives an
effect-size interval that's easy to quote directly in the report.

**Small-sample caveat:** with only a handful of evaluation slices, these tests have
limited power -- a non-significant p-value here means "not enough evidence", not
"no difference". Report the CI and effect size alongside the p-value, not the
p-value alone.

In [ ]:
diffs = (per_slice_df["unet_mean_iou"] - per_slice_df["otsu_mean_iou"]).values
n_eval = len(diffs)

wilcoxon_stat, wilcoxon_p = (np.nan, np.nan)
if n_eval >= 2 and np.any(diffs != 0):
    wilcoxon_stat, wilcoxon_p = sstats.wilcoxon(diffs)

ttest_stat, ttest_p = sstats.ttest_rel(per_slice_df["unet_mean_iou"], per_slice_df["otsu_mean_iou"])

# paired Cohen's d
cohend = diffs.mean() / diffs.std(ddof=1) if diffs.std(ddof=1) > 0 else np.nan

# bootstrap CI on the mean improvement
boot_means = np.array([
    rng.choice(diffs, size=n_eval, replace=True).mean()
    for _ in range(CONFIG["n_bootstrap"])
])
ci_low, ci_high = np.percentile(boot_means, [2.5, 97.5])

print(f"n evaluation slices: {n_eval}")
print(f"Mean IoU improvement (U-Net - Otsu): {diffs.mean():.4f}")
print(f"95% bootstrap CI on the improvement: [{ci_low:.4f}, {ci_high:.4f}]")
print(f"Paired Cohen's d: {cohend:.3f}")
print(f"Wilcoxon signed-rank: statistic={wilcoxon_stat}, p={wilcoxon_p}")
print(f"Paired t-test: t={ttest_stat:.3f}, p={ttest_p:.4f}")


## 8. Visualizations

In [ ]:
# Grouped bar chart: per-class IoU, Otsu vs U-Net
x = np.arange(len(CONFIG["phase_names"]))
width = 0.35
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.bar(x - width/2, agg_otsu["iou"], width, label="Otsu (multi-threshold)")
ax.bar(x + width/2, agg_unet["iou"], width, label="U-Net")
ax.set_xticks(x); ax.set_xticklabels(CONFIG["phase_names"])
ax.set_ylabel("IoU"); ax.set_ylim(0, 1)
ax.set_title("Per-class IoU: Otsu vs U-Net")
ax.legend()
for i, (o, u) in enumerate(zip(agg_otsu["iou"], agg_unet["iou"])):
    ax.text(i - width/2, o + 0.02, f"{o:.2f}", ha="center", fontsize=8)
    ax.text(i + width/2, u + 0.02, f"{u:.2f}", ha="center", fontsize=8)
plt.tight_layout()
plt.savefig(CONFIG["report_dir"] / "per_class_iou_bar.png", dpi=150)
plt.show()


In [ ]:
# Distribution of per-slice mean IoU across the evaluation set
fig, ax = plt.subplots(figsize=(5.5, 4.5))
box_data = [per_slice_df["otsu_mean_iou"], per_slice_df["unet_mean_iou"]]
bp = ax.boxplot(box_data, labels=["Otsu", "U-Net"], widths=0.5, patch_artist=True)
for patch, color in zip(bp["boxes"], ["#a6bddb", "#74a9cf"]):
    patch.set_facecolor(color)
for i, data in enumerate(box_data, start=1):
    jitter = rng.normal(0, 0.03, size=len(data))
    ax.scatter(np.full(len(data), i) + jitter, data, color="black", s=18, alpha=0.6, zorder=3)
ax.set_ylabel("Per-slice mean IoU")
ax.set_title(f"Per-slice mean IoU distribution (n={n_eval} slices)")
plt.tight_layout()
plt.savefig(CONFIG["report_dir"] / "per_slice_iou_boxplot.png", dpi=150)
plt.show()


In [ ]:
# Confusion matrix heatmaps (row-normalized, i.e. recall per true class)
fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))
for ax, cm, title in zip(axes, [cm_otsu_total, cm_unet_total], ["Otsu", "U-Net"]):
    cm_norm = cm / np.maximum(cm.sum(axis=1, keepdims=True), 1)
    im = ax.imshow(cm_norm, cmap="Blues", vmin=0, vmax=1)
    ax.set_xticks(range(CONFIG["n_classes"])); ax.set_xticklabels(CONFIG["phase_names"], rotation=45, ha="right")
    ax.set_yticks(range(CONFIG["n_classes"])); ax.set_yticklabels(CONFIG["phase_names"])
    ax.set_xlabel("Predicted"); ax.set_ylabel("Ground truth")
    ax.set_title(title)
    for i in range(CONFIG["n_classes"]):
        for j in range(CONFIG["n_classes"]):
            ax.text(j, i, f"{cm_norm[i, j]:.2f}", ha="center", va="center",
                     color="white" if cm_norm[i, j] > 0.5 else "black", fontsize=9)
fig.colorbar(im, ax=axes, shrink=0.8, label="fraction of true-class pixels")
plt.savefig(CONFIG["report_dir"] / "confusion_matrices.png", dpi=150, bbox_inches="tight")
plt.show()


### Qualitative comparison

Raw slice, ground truth, Otsu, and U-Net side by side for a few evaluation slices --
picks the slice where U-Net helped most and the slice where it helped least (relative
to Otsu), plus a couple more at random, so the panel shows both a best case and a
worst case rather than only flattering examples.

Mask rows use a **coloured overlay** on the raw image (voidage left untinted, membrane
in red, polymer_cartridge in cyan -- see `CONFIG["overlay_colors"]` in section 1 if you
want different colours). Set `QUALITATIVE_STYLE = "gray_labels"` in the cell below
instead for plain greyscale label maps with no colour at all.

In [ ]:
iou_gain = (per_slice_df["unet_mean_iou"] - per_slice_df["otsu_mean_iou"])
best_id = per_slice_df.loc[iou_gain.idxmax(), "sample_id"]
worst_id = per_slice_df.loc[iou_gain.idxmin(), "sample_id"]
other_ids = [sid for sid in per_slice_df["sample_id"] if sid not in (best_id, worst_id)]
extra = list(rng.choice(other_ids, size=min(2, len(other_ids)), replace=False)) if other_ids else []
show_ids = [best_id, worst_id] + extra
labels_row = (["largest U-Net gain"] + ["largest U-Net shortfall"] + ["" for _ in extra])

# QUALITATIVE_STYLE controls how the mask rows (ground truth / Otsu / U-Net) are drawn:
#   "gray_labels" -- plain greyscale label maps (each class a distinct grey level, no colour)
QUALITATIVE_STYLE = "overlay"

def render_mask_panel(ax, img, labels, style):
    if style == "gray_labels":
        # evenly-spaced grey levels 0..255 across classes, independent of label value magnitude
        n = CONFIG["n_classes"]
        grey_labels = (labels.astype(np.float32) * (255 / max(n - 1, 1)))
        ax.imshow(grey_labels, cmap="gray", vmin=0, vmax=255)
    else:
        ax.imshow(overlay(img, labels))
    ax.axis("off")

fig, axes = plt.subplots(4, len(show_ids), figsize=(3.2 * len(show_ids), 12))
if len(show_ids) == 1:
    axes = axes[:, None]
for col, sid in enumerate(show_ids):
    ex = qualitative_examples[sid]
    axes[0, col].imshow(ex["img"], cmap="gray"); axes[0, col].axis("off")
    axes[0, col].set_title(f"sample {sid}" + (f"\n({labels_row[col]})" if labels_row[col] else ""), fontsize=9)
    render_mask_panel(axes[1, col], ex["img"], ex["gt"], QUALITATIVE_STYLE)
    render_mask_panel(axes[2, col], ex["img"], ex["pred_otsu"], QUALITATIVE_STYLE)
    render_mask_panel(axes[3, col], ex["img"], ex["pred_unet"], QUALITATIVE_STYLE)
row_labels = ["raw", "ground truth", "Otsu", "U-Net"]
for r, lbl in enumerate(row_labels):
    axes[r, 0].set_ylabel(lbl, fontsize=10)
    axes[r, 0].axis("on"); axes[r, 0].set_xticks([]); axes[r, 0].set_yticks([])
    for spine in axes[r, 0].spines.values():
        spine.set_visible(False)
plt.tight_layout()
plt.savefig(CONFIG["report_dir"] / "qualitative_comparison.png", dpi=150)
plt.show()


## 9. Timing, throughput, and compute cost

Per-slice wall-clock time for each method (already captured during the timed
evaluation loop above), extrapolated to the full stack, plus model size/parameter
count and peak memory use during inference. Useful for the report's "cost of the
upgrade" section: the U-Net is presumably more accurate, but it's worth quantifying
what that costs in compute time and memory against the free-to-run Otsu baseline.

In [ ]:
time_summary = pd.DataFrame({
    "otsu_time_s": per_slice_df["otsu_time_s"],
    "unet_time_s": per_slice_df["unet_time_s"],
}).describe().loc[["mean", "std", "min", "max"]]
time_summary


In [ ]:
n_full = CONFIG["full_stack_n_slices"]
otsu_mean_s = per_slice_df["otsu_time_s"].mean()
unet_mean_s = per_slice_df["unet_time_s"].mean()

throughput_df = pd.DataFrame({
    "method": ["Otsu", "U-Net"],
    "mean_s_per_slice": [otsu_mean_s, unet_mean_s],
    "slices_per_second": [1 / otsu_mean_s, 1 / unet_mean_s],
    f"est_hours_for_{n_full}_slices": [otsu_mean_s * n_full / 3600, unet_mean_s * n_full / 3600],
})
throughput_df["slowdown_vs_otsu"] = throughput_df["mean_s_per_slice"] / otsu_mean_s
throughput_df.round(4)


### Memory and hardware

CPU figures use `psutil` (current process RSS, sampled immediately before/after each
method runs on a probe slice); GPU figures use TensorFlow's own memory-info API if a
GPU is visible. If nothing is visible, TensorFlow is running on CPU only, which is
worth stating explicitly in the report since it materially affects the timing
numbers above.

In [ ]:
def peak_rss_mb(fn, *args):
    proc = psutil.Process()
    before = proc.memory_info().rss / (1024 ** 2)
    fn(*args)
    after = proc.memory_info().rss / (1024 ** 2)
    return after, after - before

probe_row = eval_rows.iloc[0]
probe_img = load_image(sample_filename(probe_row))

otsu_rss_after, otsu_rss_delta = peak_rss_mb(segment_otsu, probe_img)
unet_rss_after, unet_rss_delta = peak_rss_mb(segment_unet, probe_img)

gpu_devices = tf.config.list_physical_devices("GPU")
gpu_mem_info = None
if gpu_devices:
    try:
        gpu_mem_info = tf.config.experimental.get_memory_info("GPU:0")
    except Exception as e:
        print(f"Could not query GPU memory info: {e}")

print(f"Process RSS after Otsu call: {otsu_rss_after:.1f} MB (delta {otsu_rss_delta:+.1f} MB)")
print(f"Process RSS after U-Net call: {unet_rss_after:.1f} MB (delta {unet_rss_delta:+.1f} MB)")
if gpu_mem_info:
    print(f"GPU memory -- current: {gpu_mem_info['current'] / (1024**2):.1f} MB, "
          f"peak: {gpu_mem_info['peak'] / (1024**2):.1f} MB")
else:
    print("No GPU visible to TensorFlow -- U-Net inference above ran on CPU.")

try:
    cpu_info = platform.processor() or platform.machine()
except Exception:
    cpu_info = "unknown"
print(f"CPU: {cpu_info}")

gpu_name = "none (CPU only)"
try:
    result = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
                             capture_output=True, text=True, timeout=5)
    if result.returncode == 0 and result.stdout.strip():
        gpu_name = result.stdout.strip().splitlines()[0]
except Exception:
    pass
print(f"GPU: {gpu_name}")


In [ ]:
compute_summary = pd.DataFrame([
    {"metric": "Model parameters", "value": f"{n_params:,}"},
    {"metric": "Model file size (MB)", "value": f"{model_size_mb:.1f}"},
    {"metric": "Otsu: mean time/slice (s)", "value": f"{otsu_mean_s:.4f}"},
    {"metric": "U-Net: mean time/slice (s)", "value": f"{unet_mean_s:.4f}"},
    {"metric": "U-Net slowdown vs. Otsu (x)", "value": f"{unet_mean_s / otsu_mean_s:.1f}"},
    {"metric": f"Otsu: est. time for {n_full} slices (h)", "value": f"{otsu_mean_s * n_full / 3600:.2f}"},
    {"metric": f"U-Net: est. time for {n_full} slices (h)", "value": f"{unet_mean_s * n_full / 3600:.2f}"},
    {"metric": "Otsu: process RSS delta (MB)", "value": f"{otsu_rss_delta:+.1f}"},
    {"metric": "U-Net: process RSS delta (MB)", "value": f"{unet_rss_delta:+.1f}"},
    {"metric": "GPU", "value": gpu_name},
    {"metric": "Training time (manual entry, hours)",
     "value": CONFIG["manual_training_time_hours"] if CONFIG["manual_training_time_hours"] is not None else "not recorded"},
])
compute_summary


## 10. Export everything for the report

In [ ]:
summary_df.round(4).to_csv(CONFIG["report_dir"] / "per_class_metrics.csv", index=False)
per_slice_df.round(4).to_csv(CONFIG["report_dir"] / "per_slice_metrics.csv", index=False)
throughput_df.round(4).to_csv(CONFIG["report_dir"] / "throughput.csv", index=False)
compute_summary.to_csv(CONFIG["report_dir"] / "compute_summary.csv", index=False)

stats_summary = pd.DataFrame([{
    "n_eval_slices": n_eval,
    "mean_iou_improvement": diffs.mean(),
    "ci_95_low": ci_low,
    "ci_95_high": ci_high,
    "cohens_d": cohend,
    "wilcoxon_statistic": wilcoxon_stat,
    "wilcoxon_p": wilcoxon_p,
    "ttest_statistic": ttest_stat,
    "ttest_p": ttest_p,
    "used_held_out_set": bool(CONFIG["held_out_sample_ids"]),
}])
stats_summary.round(5).to_csv(CONFIG["report_dir"] / "statistical_comparison.csv", index=False)

print(f"All tables and figures written to {CONFIG['report_dir']}")
for p in sorted(CONFIG["report_dir"].iterdir()):
    print(" -", p.name)


## 11. Caveats to carry into the report

- **Train/test overlap:** see the fairness note in section 0 -- if
  `held_out_sample_ids` was left empty, the U-Net numbers above are optimistic
  relative to how the model performs on genuinely new slices.
- **Small sample size:** the evaluation set is presumably a handful to a few dozen
  slices (however many hand-corrected slices exist), not thousands -- confidence
  intervals and p-values above reflect that; quote the CI, not just the point
  estimate.
- **Timing is hardware- and load-dependent:** the per-slice numbers here are wall
  clock on whatever machine ran this notebook, single-slice-at-a-time, with no other
  GPU contention. Real full-stack runs may batch differently or share the machine
  with other work -- treat the extrapolated full-stack hours as an order-of-magnitude
  estimate, not a guarantee.
- **Otsu is a fixed, global baseline.** It's the same one used to bootstrap the
  original ground truth (`xct_membrane_segmentation.ipynb`), fit once from a pooled
  histogram. A per-slice-refit Otsu might do slightly better/worse than shown here;
  this notebook deliberately mirrors the production baseline rather than an
  idealised one.
- **Disk I/O time isn't included** in the per-slice timings -- both methods load the
  slice from disk in a step outside the timed block, so these are pure
  compute/inference numbers, not end-to-end throughput including file access.
